In [1]:
import pandas as pd
import numpy as np

In [ ]:
# read enrichment data
rich = pd.read_csv('', dtype={'id_number' : str,
                                             'cellphone_number' : str})

In [ ]:
# first sheet of general support service
df1 = (
    pd.read_excel(
        io='',
        sheet_name=0,
        usecols=[1, 2, 4, 5, 6],
        skiprows=2,
        names=['umuzi_email', 'cohort_name', 'service_used', 'service_name', 'date_service_accessed']
    )
    .assign(
        umuzi_email = lambda df: df['umuzi_email'].str.split(",")
    )
    .explode("umuzi_email", ignore_index=True)
    .assign(
        umuzi_email = lambda df: df['umuzi_email'].str.strip(),
        cohort_name = lambda df: df['cohort_name'].str.strip(),
        date_service_accessed = lambda df: pd.to_datetime(df['date_service_accessed']).dt.date
    )
)

df1.head(10)

,umuzi_email,cohort_name,service_used,service_name,date_service_accessed
0,kmatulud@gmail.com,SAP Y1,Connection session,LX coach session,2026-02-25
1,dennisdaniel394@gmail.com,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01
2,lanchaster.k@gmail.com,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01
3,mashilo158@gmail.com,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01
4,erinletoluwani@gmail.com,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01
5,matsosothando@gmail.com,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01
6,badaniledanie88@gmail.com,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01
7,konanani.nemauluma@umuzi.org,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01
8,btmaluleke684@gmail.com,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01
9,alpheus.mailula@umuzi.org,Combination,Task 1: Job-Readiness and sharing of professio...,Livelihood Framework,2026-02-01


In [ ]:
df2 = (
    pd.read_excel(
        io="",
        sheet_name = 1,
        usecols=[2, 3, 5, 6, 7],
        names=['umuzi_email', 'cohort_name', 'service_used', 'service_name', 'date_service_accessed'],
        skiprows=2
    )
    .assign(
        date_service_accessed = lambda df: pd.to_datetime(df['date_service_accessed']).dt.date,
        umuzi_email = lambda df: df['umuzi_email'].str.split(",")
    )
    .explode("umuzi_email", ignore_index=True)
    .assign(
        umuzi_email = lambda df: df['umuzi_email'].str.strip()
    )
)

df2.sample(10)

,umuzi_email,cohort_name,service_used,service_name,date_service_accessed
92,ndivho.mamphiswana@umuzi.org,Wesbank PM,Connection session,LX Coach,2026-02-16
149,thembelihlengobese3@gmail.com,SAP Y2,Connection session,LX Coach,2026-02-23
94,nomfundo.mlangeni@umuzi.org,Wesbank PM,Connection session,LX Coach,2026-02-16
143,rikhotsodebra@gmail.com,SAP Y2,Connection session,LX Coach,2026-02-23
142,vilakzc@gmail.com,SAP Y2,Connection session,LX Coach,2026-02-23
73,busisiwechawane@gmail.com,SAP Y2 HCM,Connection session,LX Coach,2026-02-13
74,missnkadimeng@gmail.com,SAP Y2 HCM,Connection session,LX Coach,2026-02-13
64,dimpho.sethiba@umuzi.org,BBD A4,Connection session,LX Coach,2026-10-02
22,sibiyasa24@gmail.com,SAP Y2,Connection session,LX Coach,2026-02-02
48,sakhe.hlela@umuzi.org,BBD A4,Connection session,LX Coach,2026-06-02


In [5]:
data = pd.concat([df1, df2], ignore_index=True)

In [6]:
data.drop(
    labels=212,
    axis=0, inplace=True
)

In [7]:
data.columns

Index(['umuzi_email', 'cohort_name', 'service_used', 'service_name',
       'date_service_accessed'],
      dtype='object')

In [9]:
# Find emails in the data that can be matched on the 'email' column but not 'umuzi_email' column of the rich dataset
# This identifies records that need special handling during the merge (using email instead of umuzi_email)
a = set(set(data['umuzi_email'])).difference(set(rich['email']))
b = set(set(data['umuzi_email'])).difference(set(rich['umuzi_email']))

b - a

set()

In [10]:
len(b)

22

> Confirmation proves that the 22 are not of South African origin. They are therefore not viable to be reported on.

In [11]:
data.drop(
    labels=data.loc[data['umuzi_email'].isin(b)].index,
    axis=0, inplace=True
)

In [12]:
premerge_rows = data.shape[0]

In [13]:
data = data.merge(
    rich.drop(columns=['email']), 
    on='umuzi_email', 
    how='left')

In [14]:
assert premerge_rows == data.shape[0], "Duplicates introduced after merge due to non-unique merge keys."

In [15]:
data['learner_id'] = data['learner_id'].astype(int)

In [16]:
data['date_service_accessed'] = pd.to_datetime(data['date_service_accessed']).dt.date

In [17]:
data['age_service_accessed'] = (
    pd.to_datetime(data['date_service_accessed'])- pd.to_datetime(data['date_of_birth'])
).dt.days // 365

# add age bins
data['age_range'] = pd.cut(
    x=data['age_service_accessed'],
    bins=[-np.inf, 17, 25, 35, np.inf],
    labels=['17 and below', '18-25', '26-35', '36+']
)

In [18]:
data['month_of_service_accessed'] = pd.to_datetime(data['date_service_accessed']).dt.month_name()

data['month_of_service_accessed'] = data['month_of_service_accessed'] + ' 2025'

In [19]:
data = data[[
    'learner_id', 'application_id', 'umuzi_email',
    'first_name', 'last_name', 'cohort_name', 'cellphone_number',
    'id_number', 'gender', 'date_of_birth', 'age_service_accessed', 'age_range', 'race',
    'residential_area_type', 'has_disability_or_differently_abled',
    'application_date', 'nearest_metro', 'province', 'date_service_accessed',
    'month_of_service_accessed', 'service_used', 'service_name'
]]

In [ ]:
# export to ind7 data

# data.to_csv('../Sink Datasets/indicator_7_placements.csv', index=False)

In [ ]:
# append to existing ind7 data
# data.to_csv('../Sink Datasets/indicator_7_data.csv', index=False, mode='a', header=False)